In [4]:
# Choose the Version to Run

# Options: 'sklearn', 'scipy', 'surprise'
VERSION_TO_RUN = 'sklearn' 

print(f"Selected version: {VERSION_TO_RUN}")

Selected version: sklearn


In [5]:
#  Setup, Imports, and Initialization

import pandas as pd
import sys
import logging

# Add the source directory to the Python path
sys.path.append('src')

# Import all our custom-built classes
from database_manager import DatabaseManager
from data_loader import DataLoader

from model_builder import ModelBuilder as ModelBuilderSklearn
from recommender import Recommender as RecommenderSklearn

from model_builder_scipy import ModelBuilderSciPy
from recommender_scipy import RecommenderSciPy

# from model_builder_surprise import ModelBuilderSurprise
# from recommender_surprise import RecommenderSurprise

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

# Select classes based on the chosen version
if VERSION_TO_RUN == 'sklearn':
    ModelBuilderClass = ModelBuilderSklearn
    RecommenderClass = RecommenderSklearn
elif VERSION_TO_RUN == 'scipy':
    ModelBuilderClass = ModelBuilderSciPy
    RecommenderClass = RecommenderSciPy
elif VERSION_TO_RUN == 'surprise':
    ModelBuilderClass = ModelBuilderSurprise
    RecommenderClass = RecommenderSurprise
else:
    raise ValueError("Invalid version selected!")

# --- Initialize the System ---
db_manager = DatabaseManager()
db_manager.connect()

data_loader = DataLoader(db_manager)
dataframes = data_loader.load_all_data()

model_builder = ModelBuilderClass()
model_assets = model_builder.build_all_models(dataframes)

recommender_engine = RecommenderClass(model_assets, dataframes)

db_manager.close()

print(f"\n System is ready! Recommender engine ({VERSION_TO_RUN.upper()}) is initialized.")

2025-08-15 03:16:27,769 - database_manager - INFO - Successfully connected to DB 'movie_recommender_db'
2025-08-15 03:16:27,769 - database_manager - INFO - Session transaction isolation level set to READ COMMITTED.
2025-08-15 03:16:27,777 - data_loader - INFO - Fetching data for films...


2025-08-15 03:16:27,791 - data_loader - INFO - -> Loaded 500 rows into films_df.
2025-08-15 03:16:27,791 - data_loader - INFO - Fetching data for types...
2025-08-15 03:16:27,794 - data_loader - INFO - -> Loaded 10 rows into types_df.
2025-08-15 03:16:27,794 - data_loader - INFO - Fetching data for reviews...
2025-08-15 03:16:27,838 - data_loader - INFO - -> Loaded 5000 rows into reviews_df.
2025-08-15 03:16:27,840 - data_loader - INFO - Fetching data for clients...
2025-08-15 03:16:27,840 - data_loader - INFO - -> Loaded 200 rows into clients_df.
2025-08-15 03:16:27,840 - data_loader - INFO - Cleaning reviews_df...
2025-08-15 03:16:27,859 - data_loader - INFO - -> reviews_df shape after cleaning: (5000, 5)
2025-08-15 03:16:27,862 - model_builder - INFO - Building content-based model...
2025-08-15 03:16:27,878 - model_builder - INFO - -> Content similarity matrix created with shape: (500, 500)
2025-08-15 03:16:27,878 - model_builder - INFO - Building collaborative filtering model (skle


 System is ready! Recommender engine (SKLEARN) is initialized.


In [6]:

# Initialize the Database Manager
db_manager = DatabaseManager()
db_manager.connect()

# Initialize the Data Loader and fetch all data
data_loader = DataLoader(db_manager)
dataframes = data_loader.load_all_data()

# Initialize the Model Builder and build all model assets
model_builder = ModelBuilderSklearn()
model_assets = model_builder.build_all_models(dataframes)

# Initialize the main Recommender engine with the assets
recommender_engine = RecommenderSklearn(model_assets, dataframes)

#  Close the database connection as we have all data in memory now
db_manager.close()

print("\n System is ready! Recommender engine is initialized.")

2025-08-15 03:16:27,988 - database_manager - INFO - Successfully connected to DB 'movie_recommender_db'
2025-08-15 03:16:27,990 - database_manager - INFO - Session transaction isolation level set to READ COMMITTED.
2025-08-15 03:16:27,992 - data_loader - INFO - Fetching data for films...
2025-08-15 03:16:28,001 - data_loader - INFO - -> Loaded 500 rows into films_df.
2025-08-15 03:16:28,002 - data_loader - INFO - Fetching data for types...
2025-08-15 03:16:28,005 - data_loader - INFO - -> Loaded 10 rows into types_df.
2025-08-15 03:16:28,005 - data_loader - INFO - Fetching data for reviews...
2025-08-15 03:16:28,052 - data_loader - INFO - -> Loaded 5000 rows into reviews_df.
2025-08-15 03:16:28,052 - data_loader - INFO - Fetching data for clients...
2025-08-15 03:16:28,060 - data_loader - INFO - -> Loaded 200 rows into clients_df.
2025-08-15 03:16:28,060 - data_loader - INFO - Cleaning reviews_df...
2025-08-15 03:16:28,068 - data_loader - INFO - -> reviews_df shape after cleaning: (500


 System is ready! Recommender engine is initialized.


In [7]:
# Select a Sample User and Display Their Profile

# Let's pick a random client to generate recommendations for
sample_client = dataframes["clients"].sample(1).iloc[0]
client_id = sample_client['id']

print(f"👤 Generating recommendations for Client ID: {client_id}, Name: {sample_client['name']}\n")

# Get the reviews this client has made
client_reviews = dataframes["reviews"][dataframes["reviews"]['client_id'] == client_id]

# Merge with the films dataframe to get movie names
client_reviews_with_names = pd.merge(client_reviews, dataframes["films"], left_on='film_id', right_on='id')

print("This client has reviewed the following movies:")
# Display the user's top-rated movies
display(client_reviews_with_names[['name', 'rate']].sort_values('rate', ascending=False).head(10))

👤 Generating recommendations for Client ID: 47, Name: Kathryn Bryant

This client has reviewed the following movies:


,name,rate
17,Question Radio Here Executive Customer,4.6
11,Traditional Name Time Get,4.5
20,Former Yourself Child My Long,4.5
3,It Than Experience State,4.2
19,Meet Call Seven,4.1
1,Participant Report,4.1
16,Court Gas Hot Although,3.8
10,Kitchen Order,3.7
6,Agent Himself,3.1
0,Charge Decide Growth Trip,2.8


In [8]:
# Generate and Display the Recommendations

print(f"\n Generating hybrid recommendations for {sample_client['name']}...")

# Get a list of recommended film IDs from our engine
recommended_film_ids = recommender_engine.get_hybrid_recommendations(client_id=client_id, top_n=10)

print("\nRecommended Movies:")

if recommended_film_ids:
    # Filter the main films dataframe to get the details of the recommended movies
    recommended_films_df = dataframes["films"][dataframes["films"]['id'].isin(recommended_film_ids)]
    
    # Display the results in a clean table
    display(recommended_films_df[['id', 'name', 'type_name', 'language']])
else:
    print("Could not generate recommendations for this user.")

2025-08-15 03:16:28,196 - recommender - INFO - Diversification would be applied here.



 Generating hybrid recommendations for Kathryn Bryant...

Recommended Movies:


,id,name,type_name,language
19,20,Peace Discussion Beyond For Already,Romance,ik
51,52,Total Message,Thriller,dv
88,89,Shoulder Already Three,Horror,az
169,170,Smile Across Measure Production Mouth,Drama,sq
171,172,People Heavy North For Watch,Romance,mhr
231,232,Hospital Deep,Sci-Fi,km
251,252,Professional Within Happen,Documentary,the
300,301,Dog Vote Issue Physical Exactly,Fantasy,raj
420,421,About Discussion Analysis,Action,uz
467,468,Hundred Mission Security Field Student,Fantasy,he
